# Grad-CAM Explainability (Advanced Topic 1)

Deep-learning-only explainability for the Group 102 species classifiers. We hand-implement
Grad-CAM / Grad-CAM++ (see `grad_cam/gradcam.py`) and apply them to the trained CNNs from
`Alex_Shim/` (ResNet18/50, ConvNeXt-Tiny, Swin-T).

The four required analyses map to the four sections below:
1. **Correct vs incorrect** predictions.
2. **Confusable species pairs** (e.g. same genus).
3. **Organism vs background** attention.
4. **Concrete failure-case claims**.

**Inputs needed from teammates:** a trained `state_dict` `.pth`, the run's
`*_predictions.csv`, and the `Team_Dataset/{train,val,test}` image folders.

## Setup (Colab)

In [ ]:
# ============ Colab setup — run once ============
from google.colab import drive
drive.mount('/content/drive')

# 1) Unzip the team's prebuilt 500-class dataset from Drive
#    (built by Data Setup.ipynb -> COMP9517_Team_Subset.zip). Adjust the path if yours differs.
!unzip -q -o "/content/drive/MyDrive/COMP9517_Group_Project/COMP9517_Team_Subset.zip" -d /content/

# 2) Clone the Grad-CAM code and switch to this branch
!git clone -q https://github.com/Gr3a/COMP9517-Group-102.git /content/COMP9517-Group-102
%cd /content/COMP9517-Group-102
!git checkout feature/grad-cam

%pip install -q scikit-image

In [ ]:
import sys, os, glob
REPO = '/content/COMP9517-Group-102'
sys.path.insert(0, REPO)   # make the grad_cam package importable

import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets

from grad_cam.models import MODEL_REGISTRY, load_model
from grad_cam.gradcam import GradCAM, GradCAMpp
from grad_cam import analysis

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Auto-locate the dataset regardless of how the zip nested the folder.
DATA_DIR = next(p for p in glob.glob('/content/**/Team_Dataset', recursive=True)
                if os.path.isdir(os.path.join(p, 'test')))
print('DATA_DIR =', DATA_DIR)

In [ ]:
# ============ CONFIG — set ARCH + the two paths, then run the rest ============
ARCH         = 'resnet50'          # resnet18 | resnet50 | convnext_tiny | swin_t
NUM_CLASSES  = 500
# Point these at your chosen run's weights + predictions CSV (on Drive, or upload to Colab).
WEIGHTS_PATH = '/content/drive/MyDrive/COMP9517_Group_Project/.../r50_Pre_Full_Aug_best.pth'
PRED_CSV     = '/content/drive/MyDrive/COMP9517_Group_Project/.../r50_Pre_Full_Aug_predictions.csv'
# DATA_DIR was auto-detected in the setup cell above (Team_Dataset with train/val/test).
OUT_DIR      = 'artifacts/grad_cam'
CAM_METHOD   = GradCAM             # or GradCAMpp for sharper maps

## Smoke test (runs with no weights / data)

Confirms the hooks fire and a heatmap renders on a randomly-initialised model, before the
trained assets are plugged in. Overlay will look like noise — that is expected.

In [ ]:
_m = load_model(ARCH, NUM_CLASSES, weights_path=None, device=device)
_spec = MODEL_REGISTRY[ARCH]
with CAM_METHOD(_m, _spec.target_layer(_m), reshape=_spec.reshape) as _cam:
    _heat = _cam(torch.randn(1, 3, 224, 224, device=device), class_idx=0)
print('CAM shape', tuple(_heat.shape), 'range', float(_heat.min()), float(_heat.max()))
plt.imshow(_heat, cmap='jet'); plt.title('smoke-test CAM (random model)'); plt.axis('off'); plt.show()

## Load the trained model + predictions

In [ ]:
assert WEIGHTS_PATH and DATA_DIR and PRED_CSV, 'Fill in the CONFIG cell first.'

model = load_model(ARCH, NUM_CLASSES, weights_path=WEIGHTS_PATH, device=device)
spec = MODEL_REGISTRY[ARCH]
gradcam = CAM_METHOD(model, spec.target_layer(model), reshape=spec.reshape)

# class_names come from the alphabetical ImageFolder ordering (same as training).
class_names = datasets.ImageFolder(os.path.join(DATA_DIR, 'test')).classes
df = analysis.load_predictions(PRED_CSV)
# The CSVs store Windows paths (C:\Users\alexs\...); re-root them at the local DATA_DIR.
df = analysis.remap_filepaths(df, DATA_DIR)
df = analysis.add_genus(df, class_names)
print(len(df), 'predictions;', (df.pred == df.true).mean().round(4), 'top-1 acc')

## 1. Correct vs incorrect predictions

In [ ]:
analysis.fig_correct_vs_incorrect(gradcam, df, class_names, device=device, n=3, out_dir=OUT_DIR)
plt.show()

> **Interpretive claim (ConvNeXt-Tiny, 83.8% top-1).** On confidently correct species the CAM
> concentrates tightly on the organism's body (beetle shell, flower head, leaf rosette) with
> surrounding vegetation cold. On the misclassified examples the CAM *still* sits on the
> organism — attention is not the problem; the model localises the right object but confuses
> its fine-grained appearance with a look-alike species (developed in the sections below).

## 2. Confusable species pairs (same genus)

In [ ]:
pairs = analysis.confusable_pairs(df, class_names, top=10, same_genus_only=True)
for p in pairs:
    print(f'{p.count:3d}x  {class_names[p.true_idx]}  ->  {class_names[p.pred_idx]}')

if pairs:
    analysis.fig_confusable_pair(gradcam, df, class_names, pairs[0], device=device, out_dir=OUT_DIR)
    plt.show()

> **Interpretive claim.** Same-genus confusions split into two mechanisms. **(a) Shared
> evidence:** for the crows (*Corvus frugilegus*→*ossifragus*, CAM corr 0.93, mask IoU 0.71),
> butterflies (*Asterocampa*, 0.87/0.64) and bluebirds (*Sialia*, 0.71/0.45), the true-class
> and predicted-class CAMs overlap almost completely — the model uses the *same pixels* to
> argue for both species, so the discriminating cue is never represented. **(b) Appearance
> collapse:** for the junipers (*deppeana*→*occidentalis*, corr 0.28, IoU 0.20) the CAMs point
> at *different* foliage patches yet still confuse — every scale-leaf patch looks identical, so
> where it looks doesn't matter. Same outcome, different cause.

## 3. Organism vs background attention

In [ ]:
_, scores = analysis.fig_organism_vs_background(gradcam, df, class_names, device=device, n=4, out_dir=OUT_DIR)
plt.show()
scores  # mask_area_fraction / centroid_offset / peak_concentration (documented proxy, no GT masks)

> **Interpretive claim.** Attention locality is nearly identical for correct vs incorrect
> predictions (mask-area 0.156 vs 0.164; centroid-offset 0.213 vs 0.239, n=150 each). The CAM
> is a compact, roughly central blob (~16% of the image) in **both** cases, so the model
> attends to the organism even when wrong. Errors are therefore **not** caused by
> background/context reliance ("clever-Hans") — they are fine-grained feature confusions.

## 4. Most confident failure cases

In [ ]:
analysis.fig_failure_cases(gradcam, df, class_names, device=device, n=4, out_dir=OUT_DIR)
plt.show()
# (hooks stay attached — sections 5 and 6 below reuse this same gradcam)

> **Interpretive claim (per case).** The most confident mistakes are genus- or body-plan
> look-alikes: *Vernonia missurica*↔*gigantea* (0.96/0.92, mutual) — CAM on the identical
> purple flower head, ignoring the leaf/stem cues that separate them; *Lampropeltis
> splendida*→*calligaster* (0.92) — on shared banded scales; *Egretta rufescens*→*Ardea
> goliath* (0.91, **different genus**) — on the shared "large wading heron" silhouette; and the
> cross-family cases *Corvus albus*→*Chondrohierax uncinatus* (crow→kite, 0.91) and *Natrix
> maura*→*Agkistrodon piscivorus* (0.91) show the effect isn't limited to close relatives.
> The model relies on **coarse shared features** (flower colour, banding, body silhouette) that
> override the fine diagnostic detail. Full write-up in `grad_cam/INTERPRETATION.md`.

## 5. Faithfulness — deletion test

Blank out the most-activated CAM pixels and watch the predicted-class confidence fall. A
steep drop means the map genuinely marks the pixels the model relies on (not just a
plausible-looking heatmap). Across the three pretrained models the confidence falls by
0.33–0.45 when the top 20% of pixels are removed.

In [ ]:
curve = analysis.faithfulness_deletion(gradcam, df, device=device, n=60)
analysis.plot_deletion_curve(curve, out_dir=OUT_DIR)
plt.show()
print('confidence vs % pixels removed:', {f: round(c, 3) for f, c in curve.items()})
print('drop @20% removed =', round(curve[0.0] - curve[0.2], 3))

## 6. Attention-locality distribution (and multi-model comparison)

`background_score` over many correct vs incorrect predictions, as a distribution rather than
a single mean. Re-run this whole notebook with a different `ARCH` (resnet50, swin_t) to
compare: the CNNs give **compact** CAMs (~15% area) while **Swin-T spreads ~3× broader**
(~43%), and for **ResNet50** the incorrect predictions have visibly smaller / more off-centre
attention — a weak error-detection signal the other models don't show.

In [ ]:
dists = analysis.locality_distribution(gradcam, df, device=device, n_per=200)
analysis.plot_locality_distribution(dists, out_dir=OUT_DIR)
plt.show()
for kind, frame in dists.items():
    print(kind, frame[['mask_area_fraction', 'centroid_offset']].mean().round(3).to_dict())

gradcam.remove()  # detach hooks when finished